In [25]:
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
from typing import Optional, Tuple, Dict, Any

# ============================================================
# 1. 하이퍼파라미터 설정
# ============================================================
TEACHER_MODEL_PATH = "./model_version"  # 사전 학습된 Teacher 모델 경로
STUDENT_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
BASE_TEACHER_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # LoRA 어댑터 병합을 위한 베이스 모델명
OUTPUT_DIR = "./distilled_qwen_0.5b"
RUN_NAME = "kd_medical_distillation"

# 지식 증류 하이퍼파라미터
TEMPERATURE = 2.0  # Soft target temperature
ALPHA = 0.7        # Distillation loss weight (70% KD, 30% Ground Truth)

# ============================================================
# 2. 토크나이저 및 데이터 준비
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_dataset(data_list):
    """데이터를 Qwen 2.5 ChatML 포맷으로 변환"""
    formatted = []
    for item in data_list:
        messages = [
            {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다."},
            {"role": "user", "content": item["input"]}, 
            {"role": "assistant", "content": item["output"]},
        ]
        text = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        formatted.append({"text": text})
    return Dataset.from_list(formatted)

# 학습/평가 데이터 분할
train_dataset = format_dataset(medical_data[:25])
eval_dataset = format_dataset(medical_data[25:])

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

# ============================================================
# 3. 모델 로드 (Teacher: 사전 학습된 모델, Student + LoRA)
# ============================================================
print(f"Loading Teacher Model from {TEACHER_MODEL_PATH} (frozen)...")

# Teacher 모델 로드: 베이스 모델 + LoRA 어댑터 병합
teacher_base = AutoModelForCausalLM.from_pretrained(
    BASE_TEACHER_NAME,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
teacher_model = PeftModel.from_pretrained(teacher_base, TEACHER_MODEL_PATH)
teacher_model = teacher_model.merge_and_unload()  # LoRA 가중치를 베이스에 병합
teacher_model.eval()

# Teacher freeze (조건3)
for param in teacher_model.parameters():
    param.requires_grad = False

print(f"Loading Student Model ({STUDENT_MODEL_NAME}) with LoRA...")
student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
student_model.config.use_cache = False

# LoRA 설정 (조건4)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
student_model = get_peft_model(student_model, lora_config)
student_model.print_trainable_parameters()

# ============================================================
# 4. DistillationTrainer 구현 (조건1, 2, 5)
# ============================================================
class DistillationTrainer(SFTTrainer):
    """
    SFTTrainer을 상속받아 Knowledge Distillation Loss를 적용하는 Trainer
    """
    def __init__(self, teacher_model, temperature=2.0, alpha=0.7, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.temperature = temperature
        self.alpha = alpha
        
    def compute_loss(
        self, 
        model, 
        inputs: Dict[str, Any], 
        return_outputs: bool = False,
        num_items_in_batch: Optional[int] = None
    ) -> Tuple[torch.Tensor, Optional[Dict[str, torch.Tensor]]]:
        """
        KD Loss = α * (T² * KL_Divergence) + (1-α) * CrossEntropy
        """
        # 1. Student forward pass
        outputs = model(**inputs)
        student_logits = outputs.logits
        ce_loss = outputs.loss  # Ground Truth 기준 Cross-Entropy
        
        # 2. Teacher forward pass (no grad)
        with torch.no_grad():
            teacher_outputs = self.teacher_model(**inputs)
            teacher_logits = teacher_outputs.logits
        
        # 3. Logits 정제 (패딩 토큰 제외)
        labels = inputs.get("labels")
        if labels is not None:
            active_mask = labels != -100
            active_logits_student = student_logits[active_mask]
            active_logits_teacher = teacher_logits[active_mask]
        else:
            active_logits_student = student_logits.view(-1, student_logits.size(-1))
            active_logits_teacher = teacher_logits.view(-1, teacher_logits.size(-1))
        
        # 4. KL Divergence 계산 (Hinton et al. 공식)
        student_log_probs = F.log_softmax(active_logits_student / self.temperature, dim=-1)
        teacher_probs = F.softmax(active_logits_teacher / self.temperature, dim=-1)
        
        kl_loss = F.kl_div(
            student_log_probs, 
            teacher_probs, 
            reduction="batchmean"
        ) * (self.temperature ** 2)
        
        # 5. 최종 손실: α * KD + (1-α) * CE
        total_loss = self.alpha * kl_loss + (1 - self.alpha) * ce_loss
        
        return (total_loss, outputs) if return_outputs else total_loss

# ============================================================
# 5. 학습 설정 및 실행
# ============================================================
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
    num_train_epochs=100,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine_with_restarts",
    lr_scheduler_kwargs={"num_cycles": 3},
    warmup_steps=10,
    logging_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    gradient_checkpointing=False,
    optim="paged_adamw_8bit",
    report_to="none",
    max_grad_norm=0.3,
    max_length=1024,
)

# DistillationTrainer 인스턴스 생성 (조건5)
trainer = DistillationTrainer(
    model=student_model,
    teacher_model=teacher_model,
    temperature=TEMPERATURE,
    alpha=ALPHA,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)

# 학습 실행
separator = "=" * 50
print(f"\n{separator}")
print(f"  Knowledge Distillation Training")
print(f"  Teacher: {TEACHER_MODEL_PATH} (frozen, LoRA merged)")
print(f"  Student: {STUDENT_MODEL_NAME} (LoRA)")
print(f"  Temperature: {TEMPERATURE}, Alpha: {ALPHA}")
print(f"  Loss = {ALPHA}*(T²*KD) + {1-ALPHA}*CE")
print(separator)

result = trainer.train()

# 결과 저장
trainer.save_model(OUTPUT_DIR)
print(f"\n✅ Distillation 완료! 모델 저장: {OUTPUT_DIR}")

# 메모리 정리
del teacher_model
del student_model
del trainer
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Tokenizing eval dataset:   0%|          | 0/175 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



  Knowledge Distillation Training
  Teacher: ./model_version (frozen, LoRA merged)
  Student: Qwen/Qwen2.5-0.5B-Instruct (LoRA)
  Temperature: 2.0, Alpha: 0.7
  Loss = 0.7*(T²*KD) + 0.30000000000000004*CE


Epoch,Training Loss,Validation Loss
1,5.473441,5.384144
2,5.166979,5.016837
3,4.862716,4.735596
4,4.658823,4.377392
5,4.300550,4.143205
6,3.951180,3.966574
7,3.930662,3.801601
8,3.711523,3.667719
9,3.432401,3.565873
10,3.444311,3.492764



✅ Distillation 완료! 모델 저장: ./distilled_qwen_0.5b


In [35]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Student 모델의 베이스 모델명 (0.5B로 변경!)
STUDENT_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# 토크나이저도 Student 모델에 맞게 로드
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_and_generate(adapter_path, prompts, model_name=STUDENT_MODEL_NAME):
    """어댑터를 로드하고 추론 실행"""
    print(f"Loading base model: {model_name}")
    
    # 베이스 모델 로드 (fp16) - Student 모델인 0.5B 사용
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

    # LoRA 어댑터 로드
    print(f"Loading LoRA adapter from: {adapter_path}")
    model = PeftModel.from_pretrained(model, adapter_path)
    model.eval()

    results = []
    
    # prompts가 문자열 하나면 리스트로 변환
    if isinstance(prompts, str):
        prompts = [prompts]
    
    for prompt in prompts:
        messages = [
            {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다."},  # 한국어 시스템 프롬프트로 통일
            {"role": "user", "content": prompt},
        ]
        formatted_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.1,
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.2,
                pad_token_id=tokenizer.pad_token_id,
            )

        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        results.append(response.strip())

    del model
    torch.cuda.empty_cache()

    return results

print("추론 함수 정의 완료!")

# 테스트 실행
eval_prompts = medical_data[40]['input']

print("모델 추론 중...")
results = load_and_generate("./distilled_qwen_0.5b", eval_prompts)
print("추론 완료!\n")

print(results)

추론 함수 정의 완료!
모델 추론 중...
Loading base model: Qwen/Qwen2.5-0.5B-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading LoRA adapter from: ./distilled_qwen_0.5b
추론 완료!

['이상성 발진과 자주 발생하는 증상으로, 심장 질환이나 뇌질병의 위험도 있습니다. 24시간 콜레스테롤 검사를 받아보시는 것이 좋습니다.']
